# Pertemuan 13 — Branch and Bound dan Optimasi Kombinatorial
B&B untuk 0/1 Knapsack dan TSP vs backtracking. **[Modern]** ILP/MIP solver (SciPy `milp`, OR-Tools CP-SAT).

In [ ]:
import math, random, time, itertools
import numpy as np
random.seed(13); np.random.seed(13)

## 1. 0/1 Knapsack: backtracking vs Branch & Bound (batas = relaksasi fractional)

In [ ]:
def knap_backtrack(v, w, W):
    nodes = [0]; best = [0]
    def go(i, cap, val):
        nodes[0] += 1
        if i == len(v): best[0] = max(best[0], val); return
        if w[i] <= cap: go(i + 1, cap - w[i], val + v[i])
        go(i + 1, cap, val)
    go(0, W, 0); return best[0], nodes[0]

def knap_bnb(v, w, W):
    order = sorted(range(len(v)), key=lambda i: v[i] / w[i], reverse=True)
    V, Wt = [v[i] for i in order], [w[i] for i in order]
    def bound(i, cap, val):                                   # batas atas = greedy fractional
        for j in range(i, len(V)):
            if Wt[j] <= cap: cap -= Wt[j]; val += V[j]
            else: return val + V[j] * cap / Wt[j]
        return val
    nodes, best = [0], [0]
    def go(i, cap, val):
        nodes[0] += 1
        best[0] = max(best[0], val)
        if i == len(V) or bound(i, cap, val) <= best[0]: return   # PANGKAS
        if Wt[i] <= cap: go(i + 1, cap - Wt[i], val + V[i])
        go(i + 1, cap, val)
    go(0, W, 0); return best[0], nodes[0]

def knap_dp(v, w, W):
    dp = [0] * (W + 1)
    for a, b in zip(v, w):
        for c in range(W, b - 1, -1): dp[c] = max(dp[c], dp[c - b] + a)
    return dp[W]

print(f"{'n':>3} | {'backtrack node':>15} | {'B&B node':>9} | optimal sama?")
for n in [10, 15, 20, 24]:
    v = [random.randint(10, 100) for _ in range(n)]; w = [random.randint(5, 50) for _ in range(n)]; W = sum(w) // 3
    (a, na), (b, nb) = knap_backtrack(v, w, W), knap_bnb(v, w, W)
    print(f"{n:>3} | {na:>15} | {nb:>9} | {a == b == knap_dp(v, w, W)}")

## 2. TSP: brute force vs Branch & Bound
Batas bawah: biaya sekarang + Σ (sisi keluar termurah) untuk kota saat ini dan semua kota belum dikunjungi (tiap kota harus punya satu sisi keluar). Batas atas awal dari *nearest neighbor*.

In [ ]:
def make(n, seed):
    r = random.Random(seed); P = [(r.random(), r.random()) for _ in range(n)]
    return [[math.dist(p, q) for q in P] for p in P]

def tsp_brute(D):
    n = len(D); best = math.inf; cnt = 0
    for p in itertools.permutations(range(1, n)):
        cnt += 1; c = D[0][p[0]] + sum(D[p[i]][p[i+1]] for i in range(n-2)) + D[p[-1]][0]; best = min(best, c)
    return best, cnt

def tsp_bnb(D):
    n = len(D); min_out = [min(D[i][j] for j in range(n) if j != i) for i in range(n)]
    cur, seen, ub = 0, {0}, 0                                   # UB awal: nearest neighbor
    while len(seen) < n:
        nx = min((j for j in range(n) if j not in seen), key=lambda j: D[cur][j]); ub += D[cur][nx]; seen.add(nx); cur = nx
    best, nodes = [ub + D[cur][0]], [0]
    def go(u, visited, cost, left):
        nodes[0] += 1
        if not left: best[0] = min(best[0], cost + D[u][0]); return
        lb = cost + min_out[u] + sum(min_out[x] for x in left)
        if lb >= best[0]: return
        for x in sorted(left, key=lambda j: D[u][j]): go(x, visited | {x}, cost + D[u][x], left - {x})
    go(0, {0}, 0, frozenset(range(1, n))); return best[0], nodes[0]

print(f"{'n':>3} | {'brute (perm)':>12} {'waktu':>7} | {'B&B node':>9} {'waktu':>7} | sama?")
for n in [8, 9, 10, 12, 14]:
    D = make(n, n)
    t = time.perf_counter(); bb = tsp_bnb(D); t2 = time.perf_counter() - t
    if n <= 10:
        t = time.perf_counter(); br = tsp_brute(D); t1 = time.perf_counter() - t
        print(f"{n:>3} | {br[1]:>12} {t1:>6.2f}s | {bb[1]:>9} {t2:>6.2f}s | {abs(br[0]-bb[0])<1e-9}")
    else: print(f"{n:>3} | {'(dilewati)':>12} {'':>7} | {bb[1]:>9} {t2:>6.2f}s | -")

## 3. [Modern] Integer Linear Programming — SciPy `milp`
Model knapsack: maksimalkan Σ vᵢxᵢ, syarat Σ wᵢxᵢ ≤ W, xᵢ ∈ {0,1}. Solver MIP modern memakai B&B + *cutting planes* + heuristik.

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds
n = 40; v = np.random.randint(10, 100, n); w = np.random.randint(5, 50, n); W = int(w.sum() // 3)
t = time.perf_counter()
res = milp(c=-v, constraints=LinearConstraint(w[None, :], ub=W), integrality=np.ones(n), bounds=Bounds(0, 1))
print("MILP optimal:", int(round(-res.fun)), "| DP:", knap_dp(list(map(int, v)), list(map(int, w)), W), f"| {time.perf_counter()-t:.3f}s")

## 4. [Modern] Google OR-Tools (CP-SAT) — knapsack dan TSP
Jalankan sel instalasi di Colab (butuh internet).

In [ ]:
!pip -q install ortools

In [ ]:
from ortools.sat.python import cp_model
# --- knapsack
m = cp_model.CpModel(); x = [m.NewBoolVar(f"x{i}") for i in range(n)]
m.Add(sum(int(w[i]) * x[i] for i in range(n)) <= W); m.Maximize(sum(int(v[i]) * x[i] for i in range(n)))
s = cp_model.CpSolver(); st = s.Solve(m); print("CP-SAT knapsack:", s.StatusName(st), int(s.ObjectiveValue()))
# --- TSP dengan kendala sirkuit (jarak diskalakan jadi integer)
D = make(12, 99); N = len(D); m = cp_model.CpModel(); arcs, lits = [], {}
for i in range(N):
    for j in range(N):
        if i != j: lit = m.NewBoolVar(f"a{i}_{j}"); lits[i, j] = lit; arcs.append((i, j, lit))
m.AddCircuit(arcs); m.Minimize(sum(int(1000 * D[i][j]) * l for (i, j), l in lits.items()))
s = cp_model.CpSolver(); s.parameters.max_time_in_seconds = 10; st = s.Solve(m)
print("CP-SAT TSP:", s.StatusName(st), "panjang ≈", s.ObjectiveValue() / 1000, "| B&B kita:", round(tsp_bnb(D)[0], 3))

## 5. Latihan / Tugas
1. Ganti *bounding function* TSP dengan batas yang lebih kuat (mis. 1-tree / MST) dan bandingkan node yang dikunjungi.
2. Terapkan **best-first B&B** (priority queue berdasarkan batas) pada knapsack; bandingkan dengan DFS B&B.
3. **Tugas:** efisiensi B&B vs backtracking pada persoalan yang sama (knapsack n=15…30): tabel node, waktu, dan grafik log-skala.
4. Modelkan masalah penugasan (assignment) 8×8 di `milp` dan OR-Tools.

In [ ]:
def knap_bnb_best_first(v, w, W):
    pass  # TODO